# 05b: inference

Turns the test DICOMs into submission.csv. Runs with Internet off.

MODE = "prior" is the smoke test: every study gets 0.5 for every finding (score about 0.5),
but the whole hidden-test path runs for real: offline decoder install, scanning, series
selection with fallback, laterality, decoding and resampling every study.

Inputs: competition data, 00-dicom-decoders output, utility scripts knee_prep and knee_model.
Settings: Internet off, Environment pinned to original.
A prior submission is written first, so a later failure still leaves a valid file.

# Cell 1: imports, prior submission, offline decoders

In [ ]:
import os, sys, glob, time, subprocess
import numpy as np
import pandas as pd

ROOT = "/kaggle/input/competitions/rsna-knee-abnormality-detection"
SPLIT = "test_series"
MODE = "prior"
T0 = time.time()

for name in ["knee_prep", "knee_model"]:
    hit = glob.glob(f"/kaggle/usr/lib/**/{name}.py", recursive=True)
    print(name, hit)
    sys.path.insert(0, os.path.dirname(hit[0]))
import knee_prep as kp
import knee_model as km

test = pd.read_csv(f"{ROOT}/test.csv")
sample = pd.read_csv(f"{ROOT}/sample_submission.csv")
PRIOR = np.full(len(km.LABELS), 0.5)
km.submission_frame(test["StudyInstanceUID"], [PRIOR] * len(test)).to_csv("/kaggle/working/submission.csv", index=False)
print("prior submission written for", len(test), "studies")

def find_dir(name):
    for top in os.listdir("/kaggle/input"):
        if top == "competitions":
            continue
        hits = glob.glob(f"/kaggle/input/{top}/**/{name}", recursive=True)
        if hits:
            return hits[0]
    return None

wheels = find_dir("wheels")
if wheels:
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-index", "--find-links", wheels,
                        "pylibjpeg", "pylibjpeg-libjpeg", "pylibjpeg-openjpeg", "pylibjpeg-rle",
                        "python-gdcm"], capture_output=True, text=True)
    print("decoder install return code:", r.returncode, r.stderr[-300:])
else:
    print("decoder wheels not found")

# Cell 2: scan test series and laterality

In [ ]:
inv = None
try:
    ser = pd.read_csv(f"{ROOT}/test_series.csv")
    inv = kp.scan_all(ser, ROOT, SPLIT)
    lat = kp.study_laterality(inv)
    print(f"{len(test)} studies, {len(ser)} series, scan errors {inv['error'].notna().sum()}, "
          f"elapsed {time.time() - T0:.0f}s")
    print(lat["side_src"].value_counts().to_string())
except Exception as err:
    print("scan failed:", repr(err)[:300])

# Cell 3: decode and resample every study

In [ ]:
preds, log = {}, []
if inv is not None:
    try:
        for sid, vols, notes, sec in kp.iter_studies(inv, ROOT, SPLIT, workers=4):
            preds[sid] = PRIOR
            log.append({"StudyInstanceUID": sid, "planes": "".join(sorted(k[0] for k in vols)),
                        "n_planes": len(vols), "notes": "; ".join(notes), "sec": round(sec, 2)})
    except Exception as err:
        print("pipeline stopped:", repr(err)[:300])

log = pd.DataFrame(log, columns=["StudyInstanceUID", "planes", "n_planes", "notes", "sec"])
el = time.time() - T0
print("studies processed:", len(log), "of", len(test))
print(log["n_planes"].value_counts().to_string())
print("studies with fallback notes:", int((log["notes"] != "").sum()))
for n in log.loc[log["notes"] != "", "notes"].head(10):
    print("  ", n)
print(f"mean {log['sec'].mean():.2f}s per study per worker, total elapsed {el:.0f}s")

# Cell 4: write and check submission.csv

In [ ]:
probs = [preds.get(sid, PRIOR) for sid in test["StudyInstanceUID"]]
sub = km.submission_frame(test["StudyInstanceUID"], probs)
problems = km.check_submission(sub, sample, test["StudyInstanceUID"])
print("problems:", problems or "none")
if not problems:
    sub.to_csv("/kaggle/working/submission.csv", index=False)
print(sub.head().to_string())
print(f"total {time.time() - T0:.0f}s")